In [3]:
import pandas as pd
import numpy as np

import networkx as nx
import pickle

from pathlib import Path

from networkx.algorithms.community import louvain_communities

pd.set_option("display.max_columns", None)

In [6]:
from google.colab import drive
drive.mount('/content/drive')

!ls "/content/drive/MyDrive"

Mounted at /content/drive
'Colab Notebooks'   Dataset   Delhivery_Graph_ETA


In [8]:
!find "/content/drive/MyDrive" -name "logistics_graph.pkl"

/content/drive/MyDrive/Delhivery_Graph_ETA/data/graph_data/logistics_graph.pkl


In [12]:
GRAPH_DIR = Path(
    "/content/drive/MyDrive/Delhivery_Graph_ETA/data/graph_data"
)

with open(GRAPH_DIR / "logistics_graph.pkl", "rb") as f:
    G = pickle.load(f)


In [13]:
def graph_summary(G):

    print("="*50)
    print("GRAPH SUMMARY")
    print("="*50)

    print(f"Nodes   : {G.number_of_nodes():,}")
    print(f"Edges   : {G.number_of_edges():,}")
    print(f"Density : {nx.density(G):.6f}")

    print(
        f"Weak Components   : "
        f"{nx.number_weakly_connected_components(G)}"
    )

    print(
        f"Strong Components : "
        f"{nx.number_strongly_connected_components(G)}"
    )

graph_summary(G)

GRAPH SUMMARY
Nodes   : 1,657
Edges   : 2,783
Density : 0.001014
Weak Components   : 64
Strong Components : 545


In [15]:
degree_centrality = nx.degree_centrality(G)

in_degree_centrality = nx.in_degree_centrality(G)

out_degree_centrality = nx.out_degree_centrality(G)

In [16]:
betweenness = nx.betweenness_centrality(
    G,
    normalized=True
)

In [17]:
closeness = nx.closeness_centrality(G)

In [18]:
pagerank = nx.pagerank(
    G,
    alpha=0.85
)

HITS

In [19]:
hub_scores, authority_scores = nx.hits(
    G,
    max_iter=1000,
    normalized=True
)

In [20]:
G_undirected = G.to_undirected()

Louvain

In [21]:
communities = louvain_communities(
    G_undirected,
    seed=42
)

Community table

In [22]:
community_rows = []

for idx, community in enumerate(communities):

    size = len(community)

    for node in community:

        community_rows.append(
            {
                "facility": node,
                "community_id": idx,
                "community_size": size
            }
        )

community_df = pd.DataFrame(community_rows)

community_df.head()

,facility,community_id,community_size
0,IND673027AAB,0,43
1,IND679531AAA,0,43
2,IND686028AAB,0,43
3,IND683572AAA,0,43
4,IND670643AAA,0,43


Centrality table

In [23]:
centrality_df = pd.DataFrame({
    "facility": list(G.nodes()),

    "degree_centrality":
        [degree_centrality[n] for n in G.nodes()],

    "in_degree_centrality":
        [in_degree_centrality[n] for n in G.nodes()],

    "out_degree_centrality":
        [out_degree_centrality[n] for n in G.nodes()],

    "betweenness_centrality":
        [betweenness[n] for n in G.nodes()],

    "closeness_centrality":
        [closeness[n] for n in G.nodes()],

    "pagerank":
        [pagerank[n] for n in G.nodes()],

    "hub_score":
        [hub_scores[n] for n in G.nodes()],

    "authority_score":
        [authority_scores[n] for n in G.nodes()]
})

Graph feature store

In [24]:
graph_feature_store = centrality_df.merge(
    community_df,
    on="facility",
    how="left"
)

In [25]:
graph_feature_store.isna().sum()

,0
facility,0
degree_centrality,0
in_degree_centrality,0
out_degree_centrality,0
betweenness_centrality,0
closeness_centrality,0
pagerank,0
hub_score,0
authority_score,0
community_id,0


In [26]:
assert len(graph_feature_store) == G.number_of_nodes()

In [27]:
assert graph_feature_store["facility"].nunique() \
       == len(graph_feature_store)

In [28]:
community_df["facility"].nunique()

1657

Degree

In [29]:
graph_feature_store.nlargest(
    20,
    "degree_centrality"
)

,facility,degree_centrality,in_degree_centrality,out_degree_centrality,betweenness_centrality,closeness_centrality,pagerank,hub_score,authority_score,community_id,community_size
25,IND000000ACB,0.056763,0.027174,0.029589,0.220228,0.183963,0.011424,4.491250e-02,3.436966e-02,1,122
12,IND562132AAA,0.042874,0.021739,0.021135,0.124943,0.171499,0.009635,3.091717e-02,2.874694e-02,60,109
24,IND160002AAC,0.036836,0.019324,0.017512,0.055078,0.153548,0.008052,1.764368e-02,1.479417e-02,10,83
61,IND421302AAG,0.035024,0.017512,0.017512,0.069625,0.167048,0.006190,2.925008e-02,2.437871e-02,7,58
64,IND501359AAE,0.034420,0.018116,0.016304,0.092632,0.166713,0.008798,2.005699e-02,2.161935e-02,53,98
67,IND712311AAA,0.027778,0.014493,0.013285,0.090659,0.168480,0.006547,1.210561e-02,1.828587e-02,75,64
102,IND110037AAM,0.027174,0.012681,0.014493,0.041331,0.154629,0.004855,1.939823e-02,1.435324e-02,1,122
1,IND411033AAA,0.025966,0.013889,0.012077,0.043254,0.161800,0.006174,1.720226e-02,1.917142e-02,37,64
47,IND131028AAB,0.024155,0.012077,0.012077,0.047837,0.156051,0.005777,8.606813e-03,1.631954e-02,15,71
66,IND600056AAB,0.021739,0.010870,0.010870,0.042146,0.155466,0.004529,7.809165e-03,1.047736e-02,60,109


Betweeness

In [30]:
graph_feature_store.nlargest(
    20,
    "betweenness_centrality"
)

,facility,degree_centrality,in_degree_centrality,out_degree_centrality,betweenness_centrality,closeness_centrality,pagerank,hub_score,authority_score,community_id,community_size
25,IND000000ACB,0.056763,0.027174,0.029589,0.220228,0.183963,0.011424,0.044913,0.034370,1,122
12,IND562132AAA,0.042874,0.021739,0.021135,0.124943,0.171499,0.009635,0.030917,0.028747,60,109
64,IND501359AAE,0.034420,0.018116,0.016304,0.092632,0.166713,0.008798,0.020057,0.021619,53,98
67,IND712311AAA,0.027778,0.014493,0.013285,0.090659,0.168480,0.006547,0.012106,0.018286,75,64
61,IND421302AAG,0.035024,0.017512,0.017512,0.069625,0.167048,0.006190,0.029250,0.024379,7,58
24,IND160002AAC,0.036836,0.019324,0.017512,0.055078,0.153548,0.008052,0.017644,0.014794,10,83
47,IND131028AAB,0.024155,0.012077,0.012077,0.047837,0.156051,0.005777,0.008607,0.016320,15,71
110,IND781018AAB,0.019324,0.010266,0.009058,0.044542,0.147279,0.004665,0.004045,0.004612,82,67
1,IND411033AAA,0.025966,0.013889,0.012077,0.043254,0.161800,0.006174,0.017202,0.019171,37,64
66,IND600056AAB,0.021739,0.010870,0.010870,0.042146,0.155466,0.004529,0.007809,0.010477,60,109


Pagerank

In [31]:
graph_feature_store.nlargest(
    20,
    "pagerank"
)

,facility,degree_centrality,in_degree_centrality,out_degree_centrality,betweenness_centrality,closeness_centrality,pagerank,hub_score,authority_score,community_id,community_size
25,IND000000ACB,0.056763,0.027174,0.029589,0.220228,0.183963,0.011424,4.491250e-02,3.436966e-02,1,122
12,IND562132AAA,0.042874,0.021739,0.021135,0.124943,0.171499,0.009635,3.091717e-02,2.874694e-02,60,109
64,IND501359AAE,0.034420,0.018116,0.016304,0.092632,0.166713,0.008798,2.005699e-02,2.161935e-02,53,98
24,IND160002AAC,0.036836,0.019324,0.017512,0.055078,0.153548,0.008052,1.764368e-02,1.479417e-02,10,83
67,IND712311AAA,0.027778,0.014493,0.013285,0.090659,0.168480,0.006547,1.210561e-02,1.828587e-02,75,64
61,IND421302AAG,0.035024,0.017512,0.017512,0.069625,0.167048,0.006190,2.925008e-02,2.437871e-02,7,58
1,IND411033AAA,0.025966,0.013889,0.012077,0.043254,0.161800,0.006174,1.720226e-02,1.917142e-02,37,64
47,IND131028AAB,0.024155,0.012077,0.012077,0.047837,0.156051,0.005777,8.606813e-03,1.631954e-02,15,71
52,IND209304AAA,0.017512,0.009662,0.007850,0.040359,0.151862,0.005098,6.106367e-03,9.326770e-03,1,122
102,IND110037AAM,0.027174,0.012681,0.014493,0.041331,0.154629,0.004855,1.939823e-02,1.435324e-02,1,122


In [32]:
OUTPUT_DIR = GRAPH_DIR

centrality_df.to_csv(
    OUTPUT_DIR / "node_centrality_features.csv",
    index=False
)

community_df.to_csv(
    OUTPUT_DIR / "community_features.csv",
    index=False
)

graph_feature_store.to_csv(
    OUTPUT_DIR / "graph_feature_store.csv",
    index=False
)

In [33]:
nx.betweenness_centrality(
    G,
    k=500,
    seed=42
)

{'IND000000AAL': 0.0,
 'IND411033AAA': 0.04215679604516998,
 'IND000000AAQ': 0.0,
 'IND700028AAB': 0.0,
 'IND000000AAS': 0.004648572068608516,
 'IND783370AAC': 0.005574622356495469,
 'IND000000AAZ': 0.0008003923254363712,
 'IND444203AAA': 0.001336812598005679,
 'IND444303AAA': 0.0,
 'IND000000ABA': 0.00040120845921450156,
 'IND683565AAA': 0.0,
 'IND000000ABD': 0.0,
 'IND562132AAA': 0.1263910565028428,
 'IND000000ABG': 0.0,
 'IND501359AAF': 0.0,
 'IND000000ACA': 0.008918429003021148,
 'IND140406AAA': 0.00019979536111497918,
 'IND142001AAB': 0.0011786505538771402,
 'IND142401AAA': 0.0,
 'IND143001AAA': 0.0,
 'IND144001AAB': 0.005596374622356496,
 'IND144514AAA': 0.0004,
 'IND147301AAA': 0.00019939577039274927,
 'IND152026AAA': 0.004678036839035574,
 'IND160002AAC': 0.056826692110491725,
 'IND000000ACB': 0.21582305503794372,
 'IND000000AEL': 0.0,
 'IND110024AAA': 0.00040120845921450156,
 'IND110030AAD': 0.0,
 'IND110037AAK': 0.0,
 'IND110043AAA': 0.0017388250821885464,
 'IND110043AAC': 0.

In [34]:
community_df["community_id"].nunique()

94

In [35]:
community_df.groupby(
    "community_id"
)["facility"].count().describe()

,facility
count,94.000000
mean,17.627660
std,25.535126
min,2.000000
25%,3.000000
50%,5.000000
75%,19.750000
max,122.000000


In [36]:
community_df.groupby(
    "community_id"
)["facility"].count().sort_values(
    ascending=False
).head(20)

,facility
community_id,
1,122
60,109
53,98
10,83
22,73
15,71
82,67
37,64
75,64


In [37]:
graph_feature_store.nlargest(
    20,
    "closeness_centrality"
)

,facility,degree_centrality,in_degree_centrality,out_degree_centrality,betweenness_centrality,closeness_centrality,pagerank,hub_score,authority_score,community_id,community_size
25,IND000000ACB,0.056763,0.027174,0.029589,0.220228,0.183963,0.011424,0.044913,0.034370,1,122
12,IND562132AAA,0.042874,0.021739,0.021135,0.124943,0.171499,0.009635,0.030917,0.028747,60,109
67,IND712311AAA,0.027778,0.014493,0.013285,0.090659,0.168480,0.006547,0.012106,0.018286,75,64
61,IND421302AAG,0.035024,0.017512,0.017512,0.069625,0.167048,0.006190,0.029250,0.024379,7,58
64,IND501359AAE,0.034420,0.018116,0.016304,0.092632,0.166713,0.008798,0.020057,0.021619,53,98
68,IND751002AAB,0.011473,0.007850,0.003623,0.021587,0.162575,0.002846,0.005323,0.013295,79,29
1,IND411033AAA,0.025966,0.013889,0.012077,0.043254,0.161800,0.006174,0.017202,0.019171,37,64
62,IND462022AAA,0.013285,0.007850,0.005435,0.029638,0.159997,0.004297,0.004994,0.014986,44,42
47,IND131028AAB,0.024155,0.012077,0.012077,0.047837,0.156051,0.005777,0.008607,0.016320,15,71
66,IND600056AAB,0.021739,0.010870,0.010870,0.042146,0.155466,0.004529,0.007809,0.010477,60,109


In [38]:
graph_feature_store.nlargest(
    20,
    "hub_score"
)

,facility,degree_centrality,in_degree_centrality,out_degree_centrality,betweenness_centrality,closeness_centrality,pagerank,hub_score,authority_score,community_id,community_size
25,IND000000ACB,0.056763,0.027174,0.029589,0.220228,0.183963,0.011424,0.044913,0.034370,1,122
12,IND562132AAA,0.042874,0.021739,0.021135,0.124943,0.171499,0.009635,0.030917,0.028747,60,109
61,IND421302AAG,0.035024,0.017512,0.017512,0.069625,0.167048,0.006190,0.029250,0.024379,7,58
64,IND501359AAE,0.034420,0.018116,0.016304,0.092632,0.166713,0.008798,0.020057,0.021619,53,98
102,IND110037AAM,0.027174,0.012681,0.014493,0.041331,0.154629,0.004855,0.019398,0.014353,1,122
24,IND160002AAC,0.036836,0.019324,0.017512,0.055078,0.153548,0.008052,0.017644,0.014794,10,83
1,IND411033AAA,0.025966,0.013889,0.012077,0.043254,0.161800,0.006174,0.017202,0.019171,37,64
67,IND712311AAA,0.027778,0.014493,0.013285,0.090659,0.168480,0.006547,0.012106,0.018286,75,64
583,IND395023AAA,0.009662,0.004831,0.004831,0.013667,0.146183,0.001928,0.011188,0.007899,37,64
1022,IND560099AAB,0.017512,0.006643,0.010870,0.003475,0.138433,0.002242,0.010858,0.007739,60,109


In [39]:
graph_feature_store.nlargest(
    20,
    "authority_score"
)

,facility,degree_centrality,in_degree_centrality,out_degree_centrality,betweenness_centrality,closeness_centrality,pagerank,hub_score,authority_score,community_id,community_size
25,IND000000ACB,0.056763,0.027174,0.029589,0.220228,0.183963,0.011424,0.044913,0.034370,1,122
12,IND562132AAA,0.042874,0.021739,0.021135,0.124943,0.171499,0.009635,0.030917,0.028747,60,109
61,IND421302AAG,0.035024,0.017512,0.017512,0.069625,0.167048,0.006190,0.029250,0.024379,7,58
64,IND501359AAE,0.034420,0.018116,0.016304,0.092632,0.166713,0.008798,0.020057,0.021619,53,98
1,IND411033AAA,0.025966,0.013889,0.012077,0.043254,0.161800,0.006174,0.017202,0.019171,37,64
67,IND712311AAA,0.027778,0.014493,0.013285,0.090659,0.168480,0.006547,0.012106,0.018286,75,64
47,IND131028AAB,0.024155,0.012077,0.012077,0.047837,0.156051,0.005777,0.008607,0.016320,15,71
62,IND462022AAA,0.013285,0.007850,0.005435,0.029638,0.159997,0.004297,0.004994,0.014986,44,42
24,IND160002AAC,0.036836,0.019324,0.017512,0.055078,0.153548,0.008052,0.017644,0.014794,10,83
102,IND110037AAM,0.027174,0.012681,0.014493,0.041331,0.154629,0.004855,0.019398,0.014353,1,122
